# 02 — Data Translation
Translates the filtered English medical dataset to Bengali using
Google Translate (deep-translator), producing a bilingual dataset
for fine-tuning Swasthya on both languages.

**Input:** `cleaned_data.json` (2,567 filtered English samples)  
**Output:** `combined_data.json` (5,134 samples — English + Bengali)

## Loading Cleaned Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json

with open('/content/drive/MyDrive/swasthya/data/cleaned_data.json',
          'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")
print(data[0])

Mounted at /content/drive
Loaded 2567 samples
{'input': 'I have noticed that I ve been getting these sore reddish spots with a white ring around them on my tounge.the spot on my tounge is reddish with a white ring around it and the other 3 are bigger patches on the back of my tounge but there not red,they do have a white ring around them nd they look as if I been eating to much sour stuff which I have not.the only one that is slightly painful is the reddish one on the tip of my tounge,what could this be and should I be worried? They come and go', 'output': "Hello there, You may have a manifestation of a micronutrient deficiency. You should have a multivitamin once a day for the next 1 week preferably one with vitamin b-12(cyanocobalamine) and then assess whether your spots go down. If they don't the pleasing visit a doctor who would do tests to evaluate any underlying deficiency. Also adding probiotics to your diet like yogurt would help. Thank you for your query,"}


## Tagging English Samples

In [ ]:
english_data = []
for sample in data:
    english_data.append({
        'input': sample['input'],
        'output': sample['output'],
        'language': 'en'
    })

print(f"English samples: {len(english_data)}")
print(english_data[0])

English samples: 2567
{'input': 'I have noticed that I ve been getting these sore reddish spots with a white ring around them on my tounge.the spot on my tounge is reddish with a white ring around it and the other 3 are bigger patches on the back of my tounge but there not red,they do have a white ring around them nd they look as if I been eating to much sour stuff which I have not.the only one that is slightly painful is the reddish one on the tip of my tounge,what could this be and should I be worried? They come and go', 'output': "Hello there, You may have a manifestation of a micronutrient deficiency. You should have a multivitamin once a day for the next 1 week preferably one with vitamin b-12(cyanocobalamine) and then assess whether your spots go down. If they don't the pleasing visit a doctor who would do tests to evaluate any underlying deficiency. Also adding probiotics to your diet like yogurt would help. Thank you for your query,", 'language': 'en'}


In [ ]:
import os

save_dir = '/content/drive/MyDrive/swasthya/data'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/English_data.json', 'w',
          encoding='utf-8') as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(combined_data)} samples to Drive")

## Installing Translation Library

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00


## Setting Up Translator

In [ ]:
from deep_translator import GoogleTranslator

translator=GoogleTranslator(source='en', target='bn')

## Translating to Bengali

In [ ]:
import time
import json

bengali_data=[]
failed=0

for i, sample in enumerate(data):
  try:
    bn_input=translator.translate(sample['input'])
    time.sleep(0.3)
    bn_output=translator.translate(sample['output'])
    time.sleep(0.3)

    bengali_data.append({
        'input':bn_input,
        'output':bn_output,
        'language':'bn'
    })

  except Exception as e:
    failed += 1
    print(f"Failed at index {i}: {e}")

  if (i+1)%200==0:
    print(f"Progress:{i+1}/{len(data)}")
    print(f"Success: {len(bengali_data)} | ")
    print(f"Failed: {failed}")

print(f"\nDone!")
print(f"Translated: {len(bengali_data)}")
print(f"Failed: {failed}")

Progress:200/2567
Success: 200 | 
Failed: 0
Progress:400/2567
Success: 400 | 
Failed: 0
Progress:600/2567
Success: 600 | 
Failed: 0
Progress:800/2567
Success: 800 | 
Failed: 0
Progress:1000/2567
Success: 1000 | 
Failed: 0
Progress:1200/2567
Success: 1200 | 
Failed: 0
Progress:1400/2567
Success: 1400 | 
Failed: 0
Progress:1600/2567
Success: 1600 | 
Failed: 0
Progress:1800/2567
Success: 1800 | 
Failed: 0
Progress:2000/2567
Success: 2000 | 
Failed: 0
Progress:2200/2567
Success: 2200 | 
Failed: 0
Progress:2400/2567
Success: 2400 | 
Failed: 0

Done!
Translated: 2567
Failed: 0


## Combining English & Bengali Data

In [ ]:
# Combine both datasets
combined_data = english_data + bengali_data

print(f"English  : {len(english_data)}")
print(f"Bengali  : {len(bengali_data)}")
print(f"Combined : {len(combined_data)}")

# Quick check — one English and one Bengali sample
print("\n--- English Sample ---")
print("INPUT :", combined_data[0]['input'][:150])
print("OUTPUT:", combined_data[0]['output'][:150])

print("\n--- Bengali Sample ---")
print("INPUT :", combined_data[2567]['input'][:150])
print("OUTPUT:", combined_data[2567]['output'][:150])

English  : 2567
Bengali  : 2567
Combined : 5134

--- English Sample ---
INPUT : I have noticed that I ve been getting these sore reddish spots with a white ring around them on my tounge.the spot on my tounge is reddish with a whit
OUTPUT: Hello there, You may have a manifestation of a micronutrient deficiency. You should have a multivitamin once a day for the next 1 week preferably one 

--- Bengali Sample ---
INPUT : আমি লক্ষ্য করেছি যে আমি আমার জিভের চারপাশে একটি সাদা রিং সহ এই কালশিটে লালচে দাগ পেয়েছি। আমার জিভের দাগটি লালচে এবং তার চারপাশে একটি সাদা রিং রয়েছে 
OUTPUT: হ্যালো সেখানে, আপনার একটি মাইক্রোনিউট্রিয়েন্টের ঘাটতির প্রকাশ থাকতে পারে। আপনার পরবর্তী 1 সপ্তাহের জন্য দিনে একবার একটি মাল্টিভিটামিন খাওয়া উচিত, বি


## Saving Combined Dataset

In [ ]:
import os

save_dir = '/content/drive/MyDrive/swasthya/data'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/combined_data.json', 'w',
          encoding='utf-8') as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(combined_data)} samples to Drive")

Saved 5134 samples to Drive


## Verifying Saved Data

In [ ]:
with open(f'{save_dir}/combined_data.json', 'r',
          encoding='utf-8') as f:
    verify_data = json.load(f)

# Check a Bengali sample specifically
bengali_samples = [s for s in verify_data if s['language'] == 'bn']
print(f"Total Bengali samples: {len(bengali_samples)}")
print("\nSample Bengali input:")
print(bengali_samples[0]['input'][:200])
print("\nSample Bengali output:")
print(bengali_samples[0]['output'][:200])

Total Bengali samples: 2567

Sample Bengali input:
আমি লক্ষ্য করেছি যে আমি আমার জিভের চারপাশে একটি সাদা রিং সহ এই কালশিটে লালচে দাগ পেয়েছি। আমার জিভের দাগটি লালচে এবং তার চারপাশে একটি সাদা রিং রয়েছে এবং বাকি 3টি আমার জিভের পিছনে বড় ছোপ রয়েছে তবে ল

Sample Bengali output:
হ্যালো সেখানে, আপনার একটি মাইক্রোনিউট্রিয়েন্টের ঘাটতির প্রকাশ থাকতে পারে। আপনার পরবর্তী 1 সপ্তাহের জন্য দিনে একবার একটি মাল্টিভিটামিন খাওয়া উচিত, বিশেষত একটি ভিটামিন বি-12 (সায়ানোকোবালামাইন) সহ এবং
